# GSDM 1.1.0 on Google Colab

This notebook installs GSDM, reports the machine, runs the test suite and the benchmark, trains GVF on the
synthetic demo dataset with grouped validation, and shows the training dashboard.

**Read before choosing a runtime.** GSDM 1.1.0 runs on the CPU (pure NumPy). A GPU runtime such as an A100 is
detected by the hardware cell, but GSDM does not use it yet. A CPU runtime with high RAM gives the same results
for less. GPU support is planned for a later version.

## 1. Install

In [ ]:
!git clone --depth 1 --branch v1.1.0 https://github.com/AnThophicous/gsdm.git
%cd gsdm
!pip install -q -e ".[dev]"

## 2. What machine is this?

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "no NVIDIA GPU"
!lscpu | grep -E "Model name|^CPU\(s\)|Flags" | cut -c1-200
!free -h | head -2
import numpy as np
np.show_runtime()

## 3. Tests (gradient checks, architecture rules, end-to-end)

In [ ]:
!python -m pytest -q

## 4. Benchmark

Compare these numbers with the reference low-end machine in the README.

In [ ]:
!python benchmarks/benchmark.py

## 5. How big can a model be?

`gvf capacity` computes parameters and memory without allocating anything. The first line is a realistic size
for this task; the second shows what 10 billion parameters would need.

In [ ]:
!gvf capacity --hidden-size 256 --reasoning-steps 4
!echo
!gvf capacity --target-parameters 1e10 --reasoning-steps 4

## 6. Train on the synthetic demo dataset

The demo dataset has one group per template, so validation only contains templates the model never saw.
Expect the diagnosis to say **memorizing**: a dozen templates are not enough to learn the patterns.
That is the detector working, not a bug.

In [ ]:
!python benchmarks/synthetic_dataset.py data --variants 6
!gvf train data/manifest.jsonl --output runs/gvf.npz --epochs 80 --hidden-size 64 --reasoning-steps 4     --validation-fraction 0.3 --patience 10

## 7. Dashboard

In [ ]:
import html
from IPython.display import HTML

page = open("runs/gvf.run/dashboard.html", encoding="utf-8").read()
HTML(f'<iframe srcdoc="{html.escape(page)}" style="width:100%;height:1100px;border:0"></iframe>')

## 8. Train on your own dataset

Put your files and a `manifest.jsonl` on Google Drive. Each line looks like
`{"path": "files/a.py", "labels": ["VC-1"], "group": "project-or-cve-id"}`.
Writing the output to Drive keeps the checkpoint if Colab disconnects; run the same command with `--resume`
to continue exactly where it stopped.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET = "/content/drive/MyDrive/gvf-dataset/manifest.jsonl"
OUTPUT = "/content/drive/MyDrive/gvf-runs/gvf.npz"
!mkdir -p /content/drive/MyDrive/gvf-runs
!gvf train "$DATASET" --output "$OUTPUT" --epochs 200 --hidden-size 128 --reasoning-steps 4 --patience 15

In [ ]:
# Continue an interrupted run (same manifest, same output). --epochs is the new total.
!gvf train "$DATASET" --output "$OUTPUT" --epochs 300 --resume

## 9. Download the results

In [ ]:
!cd runs && zip -qr ../gvf-demo-run.zip .
from google.colab import files
files.download("gvf-demo-run.zip")